# Optional Week 5 extension: compare file formats

**COMPSS 211A · Fall 2026 · Student copy**

This format activity uses saved Berkeley 311 records from the HW3 data files. It introduces TSV/XML parsing with supplied code and is **outside the required relational-data lesson**.

For practice matching Monday's lesson, use [the Week 5 notebook](../lessons/week05_relational-data/week05_live_workspace.ipynb), [the pandas/SQL comparison](../docs/interactives/week05-pandas-sql.html), and [SQL practice](../docs/interactives/week05-sql-practice.html).

Work through the format tasks below if you want this extension.

## Scenario

The files describe the same set of cases, but formats can store dates, blank values, and numbers differently. Compare the loaded values before deciding whether there is a disagreement in the source records.

## Goal

- Load four formats.
- Normalize types and column order.
- Locate a representation difference and explain whether it changes the records.

## Keep handy

Use this as a self-paced format extension after the core SQL and join practice. It uses different records, field names, and parsing techniques from Monday. The code that reads JSON and XML is supplied; writing XML parsing code is not a Week 5 relational-data learning objective.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## Practice: compare four representations

Run the setup and supplied parsing cells in order. CSV separates columns with commas; TSV uses tabs; JSON stores named values; XML uses tags. The code converts each format into a DataFrame.

1. Read `format_audit`: each row is a file format; the columns count rows and distinct case IDs. Equal counts are a first check, not proof of equal contents.
2. In a new code cell, inspect `.dtypes` and a few `date_closed` values in each table. Compare the same `case_id` across files.
3. Choose one difference in how a missing value or type is stored. Explain how you would make it comparable, keeping the original tables available.

If a file is missing, check the printed data folder and that you opened the complete course repository. **Normalize** means making representations consistent, such as treating both an empty string and a missing marker as an unknown date.

In [ ]:
from xml.etree import ElementTree as ET

In [ ]:
csv_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025.csv"), dtype={"case_id": str})
tsv_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025.tsv"), sep="\t", dtype={"case_id": str})
json_rows = pd.read_json(course_data_path("hw3_campus_edge_311_2025.json"), dtype={"case_id": str})
xml_root = ET.parse(course_data_path("hw3_campus_edge_311_2025.xml")).getroot()
xml_rows = pd.DataFrame([{child.tag: child.text for child in item} for item in xml_root])
format_audit = pd.DataFrame({
    "csv": [len(csv_rows), csv_rows["case_id"].nunique()],
    "tsv": [len(tsv_rows), tsv_rows["case_id"].nunique()],
    "json": [len(json_rows), json_rows["case_id"].nunique()],
    "xml": [len(xml_rows), xml_rows["case_id"].nunique()],
}, index=["rows", "unique_cases"]).T
display(format_audit)

### Your notes

Name one case or column you compared, the difference you observed, and the check you would use after normalizing it.

> Write your notes here.

## Reflection

Which differences arose from how a format represents values? Which would require checking the source records? Explain why matching row counts alone cannot establish that two files contain equivalent data.